# Value Iteration

## Learning Objectives

By the end of this notebook, you will be able to:
- explain how value iteration updates state values directly
- implement the Bellman optimality update in a small MDP
- extract a greedy policy from the final value table
- understand why value iteration is different from policy iteration

## Prerequisites

Before starting:
- complete `02_mdp_solving.ipynb`
- understand states, actions, rewards, and terminal states
- know the difference between a policy and a value function


## Lesson Brief

This lesson focuses on **value iteration**, one of the clearest ways to compute an optimal policy in a small MDP.

Students will learn how repeated Bellman updates gradually improve value estimates until the best action becomes clear in each state.

Why this matters: value iteration gives students a concrete bridge between theory and computation.

It also prepares them to understand why later RL methods approximate values instead of calculating them exactly.

## Big idea

In policy iteration, you alternate between:
- evaluating a policy
- improving the policy

In **value iteration**, you skip full policy evaluation and update each state with the
best one-step action immediately.

This often makes value iteration simpler to teach and implement for small MDPs.

## Checkpoint Before Coding

Make sure you can explain these before continuing:

1. Why does the value of a state depend on the *next* state?
2. Why do we use `max` over actions in value iteration?
3. What should happen to terminal states during updates?

If these feel unclear, review the summary of `02_mdp_solving.ipynb` first.


In [1]:
import numpy as np

np.set_printoptions(precision=2, suppress=True)

print("=" * 70)
print("Value Iteration in a Small Grid World")
print("=" * 70)

N_ROWS, N_COLS = 3, 3
N_STATES = N_ROWS * N_COLS
ACTIONS = ["up", "right", "down", "left"]
ACTION_TO_DELTA = {
    0: (-1, 0),
    1: (0, 1),
    2: (1, 0),
    3: (0, -1),
}
ACTION_TO_ARROW = {0: "↑", 1: "→", 2: "↓", 3: "←"}
GOAL_STATE = 8
PIT_STATE = 6
TERMINAL_STATES = {GOAL_STATE, PIT_STATE}
GAMMA = 0.90


def to_pos(state):
    return divmod(state, N_COLS)


def to_state(row, col):
    return row * N_COLS + col


def transition(state, action):
    if state in TERMINAL_STATES:
        return state, 0.0

    row, col = to_pos(state)
    d_row, d_col = ACTION_TO_DELTA[action]
    next_row = min(max(row + d_row, 0), N_ROWS - 1)
    next_col = min(max(col + d_col, 0), N_COLS - 1)
    next_state = to_state(next_row, next_col)

    if next_state == GOAL_STATE:
        return next_state, 10.0
    if next_state == PIT_STATE:
        return next_state, -10.0
    return next_state, -1.0


def print_values(values):
    print("\nState values:")
    for r in range(N_ROWS):
        row = []
        for c in range(N_COLS):
            s = to_state(r, c)
            if s == GOAL_STATE:
                row.append("  G   ")
            elif s == PIT_STATE:
                row.append("  P   ")
            else:
                row.append(f"{values[s]:6.2f}")
        print(" ".join(row))


def print_policy(policy):
    print("\nGreedy policy:")
    for r in range(N_ROWS):
        row = []
        for c in range(N_COLS):
            s = to_state(r, c)
            if s == GOAL_STATE:
                row.append(" G ")
            elif s == PIT_STATE:
                row.append(" P ")
            else:
                row.append(f" {ACTION_TO_ARROW[policy[s]]} ")
        print(" ".join(row))


print("Environment ready.")
print("Goal state:", GOAL_STATE, "| Pit state:", PIT_STATE)

Value Iteration in a Small Grid World
Environment ready.
Goal state: 8 | Pit state: 6


## Part 1: One Bellman Optimality Update

For value iteration, the update rule is:

\[
V_{new}(s) = \max_a \left[ r(s,a) + \gamma V(s') \right]
\]

This means:
- try every action from the current state
- compute the return for each action
- keep the best one

We will first inspect that idea for one state before running the full algorithm.

In [2]:
values = np.zeros(N_STATES)
state = 4  # center state

print(f"Inspecting state {state} before full value iteration:\n")
action_returns = []
for action, action_name in enumerate(ACTIONS):
    next_state, reward = transition(state, action)
    target = reward + GAMMA * values[next_state]
    action_returns.append(target)
    print(
        f"Action {action_name:>5} -> next_state={next_state}, "
        f"reward={reward:>5}, target={target:>6.2f}"
    )

print("\nBest action value from state 4:", max(action_returns))
print("This is exactly the value-iteration idea: keep the best one-step return.")

Inspecting state 4 before full value iteration:

Action    up -> next_state=1, reward= -1.0, target= -1.00
Action right -> next_state=5, reward= -1.0, target= -1.00
Action  down -> next_state=7, reward= -1.0, target= -1.00
Action  left -> next_state=3, reward= -1.0, target= -1.00

Best action value from state 4: -1.0
This is exactly the value-iteration idea: keep the best one-step return.


## Part 2: Full Value Iteration

Now we run the full algorithm.

### What to watch for

- Do values near the goal become large and positive?
- Do values near the pit become very negative?
- Does the policy point around the pit instead of into it?

### Common misconception

Value iteration is **not** Q-learning.

- Value iteration assumes you know the transition model.
- Q-learning learns from interaction samples.

In Unit 1, value iteration is useful because it teaches planning clearly.

In [ ]:
def value_iteration(gamma=GAMMA, theta=1e-6):
    values = np.zeros(N_STATES)
    deltas = []

    while True:
        delta = 0.0
        new_values = values.copy()
        for state in range(N_STATES):
            if state in TERMINAL_STATES:
                continue
            action_returns = []
            for action in range(len(ACTIONS)):
                next_state, reward = transition(state, action)
                action_returns.append(reward + gamma * values[next_state])
            best_value = max(action_returns)
            delta = max(delta, abs(best_value - values[state]))
            new_values[state] = best_value
        values = new_values
        deltas.append(delta)
        if delta < theta:
            break

    return values, deltas


def greedy_policy_from_values(values, gamma=GAMMA):
    policy = np.zeros(N_STATES, dtype=int)
    for state in range(N_STATES):
        if state in TERMINAL_STATES:
            continue
        action_returns = []
        for action in range(len(ACTIONS)):
            next_state, reward = transition(state, action)
            action_returns.append(reward + gamma * values[next_state])
        policy[state] = int(np.argmax(action_returns))
    return policy


optimal_values, deltas = value_iteration()
optimal_policy = greedy_policy_from_values(optimal_values)

print_values(optimal_values)
print_policy(optimal_policy)
print(f"\nValue iteration converged in {len(deltas)} sweeps.")
print("Final Bellman update size:", deltas[-1])

print("\nSummary:")
print("- Value iteration applies the Bellman optimality update directly.")
print("- It produces optimal state values for this small known MDP.")
print("- A greedy policy can be extracted from those final values.")
print("- In the next notebook, you move from planning in a known model to working with Gym environments.")

## Closing Takeaway

**Teaching takeaway:** Value iteration works by repeatedly asking: what is the best possible return from this state if I act optimally now?

**If students remember one idea:** The Bellman optimality update is the engine of value iteration and the foundation of many later RL methods.

**Quick check before moving on:**
- Can you explain why value iteration uses a max over actions?
- Can you extract the policy idea from the learned state values?

**Bridge to the next step:** After learning how to solve small MDPs exactly, you are ready to work with interactive RL environments.
